Charger la data processed

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# charger la data processed
data_filled = pd.read_csv('../data/processed/before_06/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/before_06/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/before_06/chip_chain_log_returns.csv')

# 6 types d'indicateurs utiles

On souhaite avoir des indicateurs pour différentes parties des programmations à venir :
- HMM (Hidden Markov Model) afin d'identifier les différents états du marché
    - Bull market : faible volatilité, rendements positifs stables
    - Bear market / crise : très forte volatilité, krachs
    - Crab market : marché incertain
- LSTM (Long Short-Term Memory) pour prédire les mouvements
- Markowitz pour l'optimisation de portefeuille
- Pair trading pour l'arbitrage statistique

On créé un tableau pour stocker les 6 indicateurs

In [2]:
# créer le tableau pour les indicateurs personnalisés
features = pd.DataFrame(index=data_log_returns.index)

## Les 3 gros indicateurs

### Chip_Fear_Index La volatilité réalisée spécifique au secteur

On a déjà le VIX que l'on a télécharger qui concerne la peur globale du marché américain (S&P 500). Mais, il ne rend pas assez compte des mouvements du secteur : si une usine de TSMC brûle, le VIX va réagir trop peu par rapport à ce que l'on souhaite. Il nous faut un indice de peur sectoriel

Ainsi, on crée cela avec la volatilité réalisée (Realized Volatility), qui est l'écart-type glissant anualisé (252 jours), ici on prend une fenetre de 21 jours soit 1 mois 

In [3]:
# Créer Chip_Fear_Index : la volatilité réalisée, écart type des rendements log sur une fenêtre de 21 jours (1 mois) anualisée
numeric_df = data_log_returns.select_dtypes(include=[np.number])
volatility_21d = numeric_df.rolling(window=21).std() * np.sqrt(252) # ne pas inclure la colonne 'Date' dans le calcul de la volatilité
features['Chip_Fear_Index'] = volatility_21d[['ASML', 'TSM', '0981.HK', 'NVDA', 'AAPL']].mean(axis=1) # moyenne des volatilités des 5 actions, les plus importantes du secteur

features.shape

(3594, 1)

### Les Spreads (différentiels de rendements)

On sait qu'il existe des relations de corrélation voire une relation de reaction (c'est pourquoi le projet s'appelle ChipChainReaction)

#### Energie et matières brutes => fonderie => fabless => intégrateur (avec IA) (cf La Guerre des semi-conducteurs / Chip War)

Ainsi on souhaite mettre sur cette voie notre modèle pour qu'il comprenne ce lien, on créer donc des indiacteurs qui seront la soustraction entre deux rendements logs de tickers :
- Energie => fonderie (les couts de production)
- Fonderie => fabless (la capture des marges)
- Fabless => intégrateur (avec IA)

On ajoute aussi deux autres indicateurs de ce type :
- Le combat géopolitique entre Taiwan et la Chine : TSMC/SMIC
- Hardware contre software : Nvidia/Apple (Nvidia plus volatite que Apple)

Car $$ Spread = ln(\frac{P_{i,t}}{P_{i,t-1}}) - ln(\frac{P_{j,t}}{P_{j,t-1}}) = ln(\frac{P_{i,t}/P_{i,t-1}}{P_{j,t}/P_{j,t-1}}) = ln(\frac{P_{i,t}/P_{j,t}}{P_{i,t-1}/P_{j,t-1}}) =  ln(\frac{P_{i,t}}{P_{j,t}}) - ln(\frac{P_{i,t-1}}{P_{j,t-1}})$$

In [4]:
# 1. CRÉATION DES INDICES SECTORIELS (Paniers d'actions)
# Les matières dures (Énergie + Silicium)
features['Idx_Commodities'] = data_log_returns[['USO', 'PICK']].mean(axis=1)

# Les machines + L'usine (Les fondeurs occidentaux)
features['Idx_Western_Foundry'] = data_log_returns[['ASML', 'TSM']].mean(axis=1)

# Les concepteurs de puces (Les fabless)
features['Idx_Fabless'] = data_log_returns[['NVDA', 'AMD', 'AVGO', 'INTC']].mean(axis=1)

# Les géants de la Tech (Les acheteurs de puces)
features['Idx_Integrators'] = data_log_returns[['AAPL', 'MSFT', 'GOOGL', 'TSLA']].mean(axis=1)


# 2. CALCUL DES SPREADS MACRO (L'Effet Domino sur les secteurs entiers)
features['Spread_Macro_Commodity_Foundry'] = features['Idx_Commodities'] - features['Idx_Western_Foundry']
features['Spread_Macro_Foundry_Fabless'] = features['Idx_Western_Foundry'] - features['Idx_Fabless']
features['Spread_Macro_Fabless_Integrator'] = features['Idx_Fabless'] - features['Idx_Integrators']

# 3. LE SPREAD GÉOPOLITIQUE (Pur et isolé)
features['Spread_Geopolitics'] = data_log_returns['TSM'] - data_log_returns['0981.HK']

# 4. LE SPREAD ENTRE SOFTWARE ET HARDWARE
features['Spread_Software_Hardware'] = data_log_returns['NVDA'] - data_log_returns['AAPL']

features.shape

(3594, 10)

On a supprimé certains éléments dans les différents parties car ils sont très stables par rapport aux autres élémenrs (par exemple l'eau par rapport au pétrole)

### Les corrélations glissantes

Ici on calcule la correlation glissante c'est-à-dire $$ Corr(X_t,Y_t) = \frac{Cov(X_t,Y_t)}{\sigma_{X_t}*\sigma_{Y_t}} \quad sur\ une\ fenêtre\ mobile $$

Le but derrière est de l'exploiter pour surveiller les changements d'états (HMM), en effet, on retrouvera des corrélations pour des entreprises de même type mais lorsqu'on aura une cassure dans cette corrélation cela sera le signe d'une anomalie, une crise, des tensions géopolitiques, des problèmes de supply chain, ... Donc, en fonction de l'ampleur, un changement d'état

Le problème si on calcul toutes les corrélations glissantes : $$\binom{14}{2}=91 $$
On a :
- 91 nouvelles features (beaucoup trop)
- beaucoup de bruits
- de la redondance
- des risques d'overfitting
- un modèle plus lent
- des features sans sens

On doit donc trouver les relations importantes, c'est pourquoi j'ai choisi :
- Supply chain : ASML et TSM
- Géopolitique : TSM et 0981.HK
- Concurrence : NVDA et AMD
- Demande finale : NVDA et AAPL
- Infrastructure IA : MSFT et NVDA

In [5]:
features['Corr_ASML_TSM'] = data_log_returns['ASML'].rolling(window=63).corr(data_log_returns['TSM']) # 3 mois
features['Corr_TSM_0981'] = data_log_returns['TSM'].rolling(window=63).corr(data_log_returns['0981.HK'])
features['Corr_NVDA_AMD'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AMD'])
features['Corr_NVDA_AAPL'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AAPL'])
features['Corr_MSFT_NVDA'] = data_log_returns['MSFT'].rolling(window=63).corr(data_log_returns['NVDA'])
features.shape

(3594, 15)

## 3 autres types d'indicateurs pour détaillés au mieux les futurs modèles

### Le $\Delta TNX $ (et $\Delta VIX $)

Le TNX correspond au rendement des obligations d'Etat américaines à 10 ans (US 10-Year Treasury Yield). Cela vient du fait que le gouvernement américain emprunte de l'argent, les investisseurs achètent des obligations US considérées comme quasi sans risque ; ici le TNX mesure le rendement annuel demandé par les investisseurs pour prêter aux US pendant 10 ans

En finance, on l'utilise comme taux sans risque et référence mondiale, il influence donc les actions, Nasdaq, tech, semi-conducteurs, IA, immobiliers, dollars, presque tout

La tech y est très sensible car lorsqu'on reprend la valeur d'une action, elle dépend des profits futurs actualisés : $$ V=\sum_{t=1}^{\infty}\frac {CF_t}{(1+r)^t} \quad où\ CF_t = profits\ futurs, \ r=taux\ d'actualisation$$ ainsi si le TNX monte le coût du capital monte, la taux d'actualisation monte, les valorisation tech baisse

Ce qui nous intéresse n'est pas le TNX pur mais ces variations car ceux sont ces variations qui vont impacter le marché : $ \Delta TNX_t=TNX_t-TNX_{t-1} $ 

In [6]:
features['TNX_Diff'] = data_filled['^TNX'].diff() # Différence quotidienne du taux à 10 ans
features.shape

(3594, 16)

On fait de même pour le VIX les deux étant des indices en pourcentage

In [7]:
features['VIX_Diff'] = data_filled['^VIX'].diff() # Différence quotidienne du VIX
features.shape

(3594, 17)

### Le facteur d'inertie (Momentum/DMA)

On a pu remarqué dans nos tests de Ljung-Box (plusieurs car pour différents lags) qu'il y avait des autocorrélations souvent plus fortes plus on avait un lag élevé. Empiriquement, les marchés ont souvent de l'inertie. C'est cela que l'on veut calculer : voir si le prix actuel est anormalement loin de sa tendance récente

Pour cela on utilise : 
- une moyenne mobile
- puis on mesure l'ecart au prix actuel

La moyenne mobile (SMA : Simple Moving Average) : $$ SMA_{lag}(t)=\frac{1}{lag} \sum_{i=0}^{lag-1}P_{t-i} $$


Le DMA (Distance to Movin Average) : $$ DMA_t=\frac{P_t-SMA_{lag}(t)}{SMA_{lag}(t)} $$ Le DMA mesure à quel point le prix est éloigné de sa tendance normale

Si on regarde plus profondément dans les résultats du test de Ljung-Box d'avant on remarque que les meilleurs p-value globalement sont pour un lag de 21 et de 63 donc on calcul les deux DMA pour ces lags-ci. Et pour les momentums étudiés ont prend NVDA (leader IA), ASML (supply chain), TSM (fabrication) et AMD (concurrence GPU)

In [8]:
# NVDA
sma_21d_nvda = data_filled['NVDA'].rolling(window=21).mean()
sma_63d_nvda = data_filled['NVDA'].rolling(window=63).mean()
features['DMA_21d_NVDA'] = (data_filled['NVDA'] - sma_21d_nvda) / sma_21d_nvda
features['DMA_63d_NVDA'] = (data_filled['NVDA'] - sma_63d_nvda) / sma_63d_nvda

# ASML
sma_21d_asml = data_filled['ASML'].rolling(window=21).mean()
sma_63d_asml = data_filled['ASML'].rolling(window=63).mean()
features['DMA_21d_ASML'] = (data_filled['ASML'] - sma_21d_asml) / sma_21d_asml
features['DMA_63d_ASML'] = (data_filled['ASML'] - sma_63d_asml) / sma_63d_asml

# TSM
sma_21d_tsm = data_filled['TSM'].rolling(window=21).mean()
sma_63d_tsm = data_filled['TSM'].rolling(window=63).mean()
features['DMA_21d_TSM'] = (data_filled['TSM'] - sma_21d_tsm) / sma_21d_tsm
features['DMA_63d_TSM'] = (data_filled['TSM'] - sma_63d_tsm) / sma_63d_tsm

# AMD
sma_21d_amd = data_filled['AMD'].rolling(window=21).mean()
sma_63d_amd = data_filled['AMD'].rolling(window=63).mean()
features['DMA_21d_AMD'] = (data_filled['AMD'] - sma_21d_amd) / sma_21d_amd
features['DMA_63d_AMD'] = (data_filled['AMD'] - sma_63d_amd) / sma_63d_amd

features.shape

(3594, 25)

### Z-score / arbitrage statistique

L'idée ici c'est de repérer deux entreprises normalement proches qui deviennent éloignées anormalement. Pour cela, on calcul le Z-score : 
$$ Z_t=\frac{Spread_t-\mu _Spread}{\sigma _Spread} \qquad avec \quad Spread_t = X_t - Y_t $$
On reconnait la standardisation classique pour une loi normale. Cela calcul si le Spread est "normale", si le Z-score s'éloigne trop de 0, il y a quelque chose d'anormale (ici on parle de Spread de logarithme ce qui revient à un ratio si on sort des logarithmes)

Pour les entreprises liées ensemble, on choisit : 
- NVDA et AMD, car cce sont les grands acteurs en conception de GPU
- MSFT et AAPL, car se sont deux valeurs très stables en tech
- ASML et TSM, car TSMC est le plus gros client d'ASML et ASML est le fournisseur exclussif de TSMC (pour l'EUV)

In [9]:
def calc_zscore(serie_a, serie_b, window=63):
    spread = serie_a - serie_b
    mean = spread.rolling(window=window).mean()
    std = spread.rolling(window=window).std()
    zscore = (spread - mean) / std
    return zscore

features['Zscore_NVDA_AMD'] = calc_zscore(data_log_returns['NVDA'], data_log_returns['AMD'])
features['Zscore_MSFT_AAPL'] = calc_zscore(data_log_returns['MSFT'], data_log_returns['AAPL'])
features['Zscore_ASML_TSM'] = calc_zscore(data_log_returns['ASML'], data_log_returns['TSM'])
features.shape

(3594, 28)

Ainsi on a 28 indicateurs, tous utiles mais pas forcément pour les mêmes choses

In [10]:
features.head()

,Chip_Fear_Index,Idx_Commodities,Idx_Western_Foundry,Idx_Fabless,Idx_Integrators,Spread_Macro_Commodity_Foundry,Spread_Macro_Foundry_Fabless,Spread_Macro_Fabless_Integrator,Spread_Geopolitics,Spread_Software_Hardware,...,DMA_63d_NVDA,DMA_21d_ASML,DMA_63d_ASML,DMA_21d_TSM,DMA_63d_TSM,DMA_21d_AMD,DMA_63d_AMD,Zscore_NVDA_AMD,Zscore_MSFT_AAPL,Zscore_ASML_TSM
0,NaN,0.011711,0.017455,0.015889,0.016980,-0.005744,0.001567,-0.001091,0.032743,0.011111,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-0.004805,-0.008881,-0.013783,0.012447,0.004076,0.004901,-0.026230,0.054909,-0.016904,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,0.007089,0.004249,0.009393,0.001312,0.002839,-0.005143,0.008080,-0.019818,-0.007876,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,0.006326,0.006727,0.017189,0.010555,-0.000401,-0.010462,0.006634,-0.036605,0.018968,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-0.000962,0.004705,0.001380,0.015095,-0.005666,0.003325,-0.013715,0.006323,-0.034622,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Certains ont un début vide car ils ont des rolling(63), les 63 premières lignes du DataFrameont plusieurs NaN. Donc on doit clean ça en les supprimant

In [11]:
features_clean = features.dropna()
features_clean.shape

(3532, 28)

Parfait on a bien 63 lignes en moins, il ne reste plus qu'à le sauvegarder

In [12]:
features_clean.to_csv('../data/processed/before_06/chip_chain_custom_indicators.csv', index=False)

In [13]:
# refaire les calculs des indicateurs personnalisés pour les données après la séance 06 et non before_06
data_filled = pd.read_csv('../data/processed/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/chip_chain_log_returns.csv')

features = pd.DataFrame(index=data_log_returns.index)
features['Chip_Fear_Index'] = data_log_returns[['ASML', 'TSM', '0981.HK', 'NVDA', 'AAPL']].rolling(window=21).std().mean(axis=1) * np.sqrt(252) 

features['Idx_Commodities'] = data_log_returns[['USO', 'PICK']].mean(axis=1)
features['Idx_Western_Foundry'] = data_log_returns[['ASML', 'TSM']].mean(axis=1)
features['Idx_Fabless'] = data_log_returns[['NVDA', 'AMD', 'AVGO', 'INTC']].mean(axis=1)
features['Idx_Integrators'] = data_log_returns[['AAPL', 'MSFT', 'GOOGL', 'TSLA']].mean(axis=1)
# on ajoute un index rassemblant GLD et TLT qui n'est pas dans les données before_06
features['Idx_Safe_Haven'] = data_log_returns[['GLD', 'TLT']].mean(axis=1)

features['Spread_Macro_Commodity_Foundry'] = features['Idx_Commodities'] - features['Idx_Western_Foundry']
features['Spread_Macro_Foundry_Fabless'] = features['Idx_Western_Foundry'] - features['Idx_Fabless']
features['Spread_Macro_Fabless_Integrator'] = features['Idx_Fabless'] - features['Idx_Integrators']

features['Spread_Geopolitics'] = data_log_returns['TSM'] - data_log_returns['0981.HK']
features['Spread_Software_Hardware'] = data_log_returns['NVDA'] - data_log_returns['AAPL']
# on ajoute un spread entre les safe havens et les intégrateurs pour voir si les intégrateurs sont plus ou moins risqués que les safe havens
features['Spread_Safe_Haven_Integrators'] = features['Idx_Safe_Haven'] - features['Idx_Integrators']

features['Corr_ASML_TSM'] = data_log_returns['ASML'].rolling(window=63).corr(data_log_returns['TSM'])
features['Corr_TSM_0981'] = data_log_returns['TSM'].rolling(window=63).corr(data_log_returns['0981.HK'])
features['Corr_NVDA_AMD'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AMD'])
features['Corr_NVDA_AAPL'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AAPL'])
features['Corr_MSFT_NVDA'] = data_log_returns['MSFT'].rolling(window=63).corr(data_log_returns['NVDA'])

features['TNX_Diff'] = data_filled['^TNX'].diff()
features['VIX_Diff'] = data_filled['^VIX'].diff()


sma_21d_nvda = data_filled['NVDA'].rolling(window=21).mean()
sma_63d_nvda = data_filled['NVDA'].rolling(window=63).mean()
features['DMA_21d_NVDA'] = (data_filled['NVDA'] - sma_21d_nvda) / sma_21d_nvda
features['DMA_63d_NVDA'] = (data_filled['NVDA'] - sma_63d_nvda) / sma_63d_nvda

sma_21d_asml = data_filled['ASML'].rolling(window=21).mean()
sma_63d_asml = data_filled['ASML'].rolling(window=63).mean()
features['DMA_21d_ASML'] = (data_filled['ASML'] - sma_21d_asml) / sma_21d_asml
features['DMA_63d_ASML'] = (data_filled['ASML'] - sma_63d_asml) / sma_63d_asml

sma_21d_tsm = data_filled['TSM'].rolling(window=21).mean()
sma_63d_tsm = data_filled['TSM'].rolling(window=63).mean()
features['DMA_21d_TSM'] = (data_filled['TSM'] - sma_21d_tsm) / sma_21d_tsm
features['DMA_63d_TSM'] = (data_filled['TSM'] - sma_63d_tsm) / sma_63d_tsm

sma_21d_amd = data_filled['AMD'].rolling(window=21).mean()
sma_63d_amd = data_filled['AMD'].rolling(window=63).mean()
features['DMA_21d_AMD'] = (data_filled['AMD'] - sma_21d_amd) / sma_21d_amd
features['DMA_63d_AMD'] = (data_filled['AMD'] - sma_63d_amd) / sma_63d_amd

features['Zscore_NVDA_AMD'] = calc_zscore(data_log_returns['NVDA'], data_log_returns['AMD'])
features['Zscore_MSFT_AAPL'] = calc_zscore(data_log_returns['MSFT'], data_log_returns['AAPL'])
features['Zscore_ASML_TSM'] = calc_zscore(data_log_returns['ASML'], data_log_returns['TSM'])
features_clean = features.dropna()
print(features_clean.shape)
features_clean.to_csv('../data/processed/chip_chain_custom_indicators.csv', index=False)

(3532, 30)
